# Baseline content-based movie recommendations

This notebook recommends movies with similar **genres**. Content-based filtering compares item attributes: if you choose a movie, we find other movies whose attributes resemble it. This baseline uses one selected movie; it does not yet learn a user's personal preferences or use other users' ratings.

We use the already processed CSV from notebook 01. We only create model features here; we do not rewrite the dataset.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load the processed movies

The path below works when Jupyter starts from either the project root or the `notebooks` folder. `movieId` is our unique identifier. A title is only a search label because different records can have the same title.

In [5]:
data_path = Path("data/processed/movies_processed.csv")
if not data_path.exists():
    data_path = Path("../data/processed/movies_processed.csv")

movies = pd.read_csv(data_path)
required_columns = ["movieId", "title", "genres"]
if not set(required_columns).issubset(movies.columns):
    raise ValueError("The processed CSV must contain movieId, title, and genres.")
movies = movies[required_columns].copy()
if movies[required_columns].isna().any().any():
    raise ValueError("The processed CSV contains missing IDs, titles, or genres.")
if not movies["movieId"].is_unique:
    raise ValueError("movieId must uniquely identify each movie.")

movies = movies.set_index("movieId")
print(f"Loaded {len(movies):,} movies; movieId is the unique index.")
display(movies.head())

Loaded 9,742 movies; movieId is the unique index.


,title,genres
movieId,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy


In [6]:
duplicate_titles = movies[movies["title"].duplicated(keep=False)].sort_values("title")
print(f"Duplicated titles: {duplicate_titles['title'].nunique()}")
display(duplicate_titles.reset_index())

Duplicated titles: 5


,movieId,title,genres
0,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
1,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
2,838,Emma (1996),Comedy|Drama|Romance
3,26958,Emma (1996),Romance
4,32600,Eros (2004),Drama
5,147002,Eros (2004),Drama|Romance
6,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
7,168358,Saturn 3 (1980),Sci-Fi|Thriller
8,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
9,64997,War of the Worlds (2005),Action|Sci-Fi


## 2. Turn genres into numerical features

Genres are understandable content features shared by many movies. A movie can have several genres, so we split each genre string at `|` and use **MultiLabelBinarizer**. It learns one column per genre and assigns **1** when a movie has that genre and **0** otherwise. These numbers indicate membership, not a ranking of genres.

`(no genres listed)` means the genre information is unavailable. We replace that marker with an empty list in the model features, producing an all-zero row rather than treating missing information as a real genre. The original `genres` column stays intact.

In [7]:
genre_lists = movies["genres"].str.split("|")
genre_lists = genre_lists.apply(
    lambda genres: [genre for genre in genres if genre != "(no genres listed)"]
)

encoder = MultiLabelBinarizer()
genre_features = pd.DataFrame(
    encoder.fit_transform(genre_lists),
    index=movies.index,
    columns=encoder.classes_,
)

print(f"Feature matrix: {genre_features.shape[0]:,} movies × {genre_features.shape[1]} genres")
print(f"Movies without usable genres: {(genre_features.sum(axis=1) == 0).sum()}")
display(genre_features.head(10))

Feature matrix: 9,742 movies × 19 genres
Movies without usable genres: 34


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movieId,,,,,,,,,,,,,,,,,,,
1,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
7,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
8,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Each matrix row is identified by `movieId`, so duplicate titles cannot merge rows. For example, Toy Story has a 1 for Animation and Comedy, but a 0 for Horror. The table below puts the movie label next to its binary features for inspection.

In [8]:
display(movies[["title", "genres"]].join(genre_features).head())

,title,genres,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movieId,,,,,,,,,,,,,,,,,,,,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Jumanji (1995),Adventure|Children|Fantasy,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,Waiting to Exhale (1995),Comedy|Drama|Romance,0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
5,Father of the Bride Part II (1995),Comedy,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Compare genre vectors with cosine similarity

Cosine similarity measures the angle between two feature vectors. For these nonnegative binary features, scores range from **0** (no shared genres, or a zero vector) to **1** (the same genre set). A shared genre contributes to similarity, while the vector lengths account for how many genres each movie has.

For example, `[1, 1, 0]` and `[1, 0, 0]` share one genre and have a similarity of about 0.707. That score is a measure of feature overlap, not a probability that a user will like the movie.

We calculate one selected movie's similarity to every movie when needed. This avoids storing a large movie-by-movie matrix and uses the same cosine calculation.

In [9]:
example_id = movies.index[0]
example_scores = cosine_similarity(
    genre_features.loc[[example_id]], genre_features
).ravel()

similarity_preview = movies.reset_index().copy()
similarity_preview["similarity_score"] = example_scores
print(f"Similarity to: {movies.loc[example_id, 'title']} (movieId={example_id})")
display(similarity_preview.head(10))

Similarity to: Toy Story (1995) (movieId=1)


,movieId,title,genres,similarity_score
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.000000
1,2,Jumanji (1995),Adventure|Children|Fantasy,0.774597
2,3,Grumpier Old Men (1995),Comedy|Romance,0.316228
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,0.258199
4,5,Father of the Bride Part II (1995),Comedy,0.447214
5,6,Heat (1995),Action|Crime|Thriller,0.000000
6,7,Sabrina (1995),Comedy|Romance,0.316228
7,8,Tom and Huck (1995),Adventure|Children,0.632456
8,9,Sudden Death (1995),Action,0.000000
9,10,GoldenEye (1995),Action|Adventure|Thriller,0.258199


## 4. Recommend movies by title

`recommend_movies(title)` performs an exact title match, ignoring capitalization and surrounding spaces. Include the year as it appears in the dataset. An unknown title raises a clear error.

If multiple records match, the function lists their IDs and asks you to pass `movie_id` too. The selected ID must belong to that title. All feature selection and self-exclusion then use `movieId`.

The function computes cosine scores, removes the selected movie, sorts by descending score, and returns the top 10. Ties are ordered by ascending `movieId` to make results repeatable; this tie order does not imply a preference. A selected movie without genres raises an error because it cannot provide a meaningful genre comparison. Other movies without genres receive score 0 and rank below positive matches.

In [11]:
def recommend_movies(title, movie_id=None):
    """Return 10 genre-based matches; use movie_id to resolve duplicate titles."""
    if not isinstance(title, str) or not title.strip():
        raise ValueError("Enter a nonempty movie title, including its year.")

    matches = movies[
        movies["title"].str.strip().str.casefold() == title.strip().casefold()
    ]
    if matches.empty:
        raise ValueError(f"No movie found for {title!r}. Check the title and year.")

    if movie_id is None:
        if len(matches) > 1:
            choices = matches.reset_index().to_string(index=False)
            raise ValueError(
                "Multiple movies have this title. Pass movie_id from this table:\n"
                + choices
            )
        selected_id = matches.index[0]
    else:
        if movie_id not in matches.index:
            raise ValueError("movie_id must identify a movie with the supplied title.")
        selected_id = movie_id

    selected_features = genre_features.loc[[selected_id]]
    if selected_features.to_numpy().sum() == 0:
        raise ValueError("This movie has no listed genres; choose another movie.")

    scores = cosine_similarity(selected_features, genre_features).ravel()
    results = movies.reset_index().copy()
    results["similarity_score"] = scores
    results = results[results["movieId"] != selected_id]
    results = results.sort_values(
        ["similarity_score", "movieId"], ascending=[False, True]
    )
    return results.head(10).reset_index(drop=True)

## 5. Try several movies

These titles come from the loaded dataset. Each output includes the recommended movie's unique ID, title, original genres, and cosine score. Several movies can tie at 1.0 because their genre sets are identical, even if their stories are very different.

In [12]:
example_titles = [
    "Toy Story (1995)",
    "Jumanji (1995)",
    "Grumpier Old Men (1995)",
]
for title in example_titles:
    print(f"Recommendations for {title}")
    display(recommend_movies(title).round({"similarity_score": 3}))

Recommendations for Toy Story (1995)


,movieId,title,genres,similarity_score
0,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
5,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
7,65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
8,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
9,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0


Recommendations for Jumanji (1995)


,movieId,title,genres,similarity_score
0,60,"Indian in the Cupboard, The (1995)",Adventure|Children|Fantasy,1.0
1,126,"NeverEnding Story III, The (1994)",Adventure|Children|Fantasy,1.0
2,1009,Escape to Witch Mountain (1975),Adventure|Children|Fantasy,1.0
3,2043,Darby O'Gill and the Little People (1959),Adventure|Children|Fantasy,1.0
4,2093,Return to Oz (1985),Adventure|Children|Fantasy,1.0
5,2161,"NeverEnding Story, The (1984)",Adventure|Children|Fantasy,1.0
6,2162,"NeverEnding Story II: The Next Chapter, The (1...",Adventure|Children|Fantasy,1.0
7,2399,Santa Claus: The Movie (1985),Adventure|Children|Fantasy,1.0
8,4896,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy,1.0
9,41566,"Chronicles of Narnia: The Lion, the Witch and ...",Adventure|Children|Fantasy,1.0


Recommendations for Grumpier Old Men (1995)


,movieId,title,genres,similarity_score
0,7,Sabrina (1995),Comedy|Romance,1.0
1,39,Clueless (1995),Comedy|Romance,1.0
2,64,Two if by Sea (1996),Comedy|Romance,1.0
3,68,French Twist (Gazon maudit) (1995),Comedy|Romance,1.0
4,118,If Lucy Fell (1996),Comedy|Romance,1.0
5,122,Boomerang (1992),Comedy|Romance,1.0
6,129,Pie in the Sky (1996),Comedy|Romance,1.0
7,180,Mallrats (1995),Comedy|Romance,1.0
8,186,Nine Months (1995),Comedy|Romance,1.0
9,237,Forget Paris (1995),Comedy|Romance,1.0


### Explicitly resolve a duplicate title

First show the ambiguity message. Then make a recommendation request for each matching record using its `movieId`. Records sharing a title may also share genres, so their results can look alike. Only the selected ID is excluded; another record with the same title remains a separate candidate.

In [13]:
duplicate_title = duplicate_titles["title"].iloc[0]
try:
    recommend_movies(duplicate_title)
except ValueError as error:
    print(error)

for movie_id, movie in movies[movies["title"] == duplicate_title].iterrows():
    print(f"Recommendations for {movie['title']} (movieId={movie_id})")
    display(recommend_movies(movie["title"], movie_id=movie_id).round({"similarity_score": 3}))

Multiple movies have this title. Pass movie_id from this table:
 movieId                                  title                              genres
    6003 Confessions of a Dangerous Mind (2002)         Comedy|Crime|Drama|Thriller
  144606 Confessions of a Dangerous Mind (2002) Comedy|Crime|Drama|Romance|Thriller
Recommendations for Confessions of a Dangerous Mind (2002) (movieId=6003)


,movieId,title,genres,similarity_score
0,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,1.000
1,608,Fargo (1996),Comedy|Crime|Drama|Thriller,1.000
2,1034,Freeway (1996),Comedy|Crime|Drama|Thriller,1.000
3,3266,Man Bites Dog (C'est arrivé près de chez vous)...,Comedy|Crime|Drama|Thriller,1.000
4,4242,Beautiful Creatures (2000),Comedy|Crime|Drama|Thriller,1.000
5,6705,Party Monster (2003),Comedy|Crime|Drama|Thriller,1.000
6,57669,In Bruges (2008),Comedy|Crime|Drama|Thriller,1.000
7,71211,"Informant!, The (2009)",Comedy|Crime|Drama|Thriller,1.000
8,75813,Leaves of Grass (2009),Comedy|Crime|Drama|Thriller,1.000
9,20,Money Train (1995),Action|Comedy|Crime|Drama|Thriller,0.894


Recommendations for Confessions of a Dangerous Mind (2002) (movieId=144606)


,movieId,title,genres,similarity_score
0,1912,Out of Sight (1998),Comedy|Crime|Drama|Romance|Thriller,1.000
1,3893,Nurse Betty (2000),Comedy|Crime|Drama|Romance|Thriller,1.000
2,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,0.894
3,496,What Happened Was... (1994),Comedy|Drama|Romance|Thriller,0.894
4,608,Fargo (1996),Comedy|Crime|Drama|Thriller,0.894
5,866,Bound (1996),Crime|Drama|Romance|Thriller,0.894
6,1034,Freeway (1996),Comedy|Crime|Drama|Thriller,0.894
7,2166,Return to Paradise (1998),Crime|Drama|Romance|Thriller,0.894
8,2764,"Thomas Crown Affair, The (1968)",Crime|Drama|Romance|Thriller,0.894
9,3266,Man Bites Dog (C'est arrivé près de chez vous)...,Comedy|Crime|Drama|Thriller,0.894


### Basic correctness checks

These checks verify behavior, not recommendation quality: ten results, unique IDs, valid and sorted scores, correct self-exclusion, case-insensitive matching, and clear errors for invalid requests.

In [14]:
for title in example_titles:
    recommendations = recommend_movies(title)
    selected_id = movies.index[movies["title"] == title][0]
    assert len(recommendations) == 10
    assert recommendations["movieId"].is_unique
    assert selected_id not in recommendations["movieId"].values
    assert recommendations["similarity_score"].between(-1e-12, 1 + 1e-12).all()
    assert recommendations["similarity_score"].is_monotonic_decreasing

pd.testing.assert_frame_equal(
    recommend_movies("  toy story (1995)  "), recommend_movies("Toy Story (1995)")
)
for movie_id in duplicate_titles.index:
    title = movies.loc[movie_id, "title"]
    assert movie_id not in recommend_movies(title, movie_id)["movieId"].values

# Confirm identical genre vectors have cosine similarity 1.
assert abs(example_scores[0] - 1.0) < 1e-12

def check_clear_error(title, movie_id=None):
    try:
        recommend_movies(title, movie_id)
    except ValueError as error:
        print(error)
    else:
        raise AssertionError("Expected a clear ValueError for this request.")

check_clear_error("A title that does not exist in this dataset")
check_clear_error("Toy Story (1995)", movie_id=-1)
no_genre_movies = movies[genre_features.sum(axis=1) == 0]
if not no_genre_movies.empty:
    movie_id = no_genre_movies.index[0]
    check_clear_error(movies.loc[movie_id, "title"], movie_id)
print("All correctness checks passed.")

No movie found for 'A title that does not exist in this dataset'. Check the title and year.
movie_id must identify a movie with the supplied title.
This movie has no listed genres; choose another movie.
All correctness checks passed.


## Limitations and future evaluation

- Genres are broad: movies with identical genres receive identical similarity scores even if their plots, tone, or quality differ. Scores do not measure enjoyment or accuracy.
- Every genre is weighted equally. We do not use plots, cast, directors, release dates, popularity, or user ratings.
- This is a movie-to-movie baseline, not a learned user profile. It cannot capture individual taste or recommend meaningfully from a movie with no known genres.
- Duplicate IDs are rejected, but distinct IDs with the same title remain separate candidates. A separate data review would be needed to determine whether those records describe the same film.
- If fewer than ten candidates share a genre, zero-score candidates can fill the remaining positions. A minimum-similarity rule could be explored later.

The examples and assertions demonstrate behavior; **no recommendation performance has been measured**. To measure personalized quality, we would need user interaction/rating data, a justified definition of relevance, and a train/test split that prevents held-out interactions from informing user profiles or seed selection. We would also need to exclude already-seen movies and define the candidate set consistently.

With that setup, Precision@10 would measure the fraction of recommended movies that are relevant; Recall@10 would measure the fraction of held-out relevant movies retrieved; NDCG@10 would account for their rank and optionally graded relevance. Coverage and diversity could describe how broadly the system recommends. A user study could also assess whether users find suggestions useful. None of these quality metrics can be inferred from genre similarity scores alone.